# 🚀 SPEC-03 (v6 - Single Best Model Storage Protection)
## 📌 IndoRoBERTa-base (110M) vs. XLM-RoBERTa-large (550M)

- **Target Models:**
  1. **IndoRoBERTa-base** (`indolem/indobert-base-uncased`, 110M Monolingual) — 5-Fold Stratified CV, 20 Epochs
  2. **XLM-RoBERTa-large** (`xlm-roberta-large`, 550M Multilingual Target) — 5-Fold Stratified CV + Expanded Head (`2048 -> 512 -> 4`), 20 Epochs
- **Key Storage Safeguard v6:**
  - **Single Best Model Storage Safeguard:** Tracks the single best performing checkpoint across all 5 folds and saves only **1 best model file (~2.2 GB)** instead of 5 files (11 GB), saving over **8.8 GB** of Kaggle disk storage!
  - **5-Fold Stratified Cross-Validation:** Full 954 gold-standard dataset evaluation ($Mean \pm Std$).
  - **Real-Time Val F1 Progress Display:** Real-time logging of Train Loss, Val F1 (Active), Val F1 (All), and Val Acc per epoch.
  - **No Data Augmentation:** 100% pure human gold-standard text.
  - **Aspect-Customized Thresholding ($	heta_{	ext{aspect}}$):** `purnajual`: 0.35, `infra`: 0.40, `ekonomi`: 0.50, `kualitas`: 0.50.


## 1. Setup Dependencies & Environment Imports

In [ ]:
# Install required packages for Kaggle T4 GPU
!pip install -q -U transformers accelerate scikit-learn seaborn matplotlib tqdm

import os
import gc
import json
import zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import get_cosine_schedule_with_warmup, AutoModel, AutoTokenizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

# Check CUDA availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"⚡ Device Utama: {device}")
if torch.cuda.is_available():
    print(f"🎮 GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"💾 Total VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

# Set Random Seed for Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

LABEL2ID = {'None': 0, 'none': 0, 'positif': 1, 'netral': 2, 'negatif': 3}
ID2LABEL = {0: 'None', 1: 'positif', 2: 'netral', 3: 'negatif'}
ASPECTS = ['infra', 'ekonomi', 'kualitas', 'purnajual']
CLASS_NAMES = ['None', 'Positif', 'Netral', 'Negatif']

# Aspect-customized decision thresholds
ASPECT_THRESHOLDS = {
    "infra": 0.40,
    "ekonomi": 0.50,
    "kualitas": 0.50,
    "purnajual": 0.35,
}


## 2. Load Full Gold Standard Dataset (954 Rows)

In [ ]:
# Search for dataset file (valid_labeled_comments.csv or train.csv + val.csv)
data_dir = None
possible_paths = [
    Path("/kaggle/input/indonesian-ev-absa-dataset"),
    Path("/kaggle/input"),
    Path("data/interim"),
    Path("data/processed"),
    Path("../data/processed")
]

full_df = None
for p in possible_paths:
    if p.exists():
        if (p / "valid_labeled_comments.csv").exists():
            full_df = pd.read_csv(p / "valid_labeled_comments.csv", encoding="utf-8-sig")
            data_dir = p
            break
        elif (p / "train.csv").exists() and (p / "val.csv").exists():
            df_tr = pd.read_csv(p / "train.csv", encoding="utf-8-sig")
            df_va = pd.read_csv(p / "val.csv", encoding="utf-8-sig")
            full_df = pd.concat([df_tr, df_va], ignore_index=True)
            data_dir = p
            break
        # Search subdirectories inside /kaggle/input
        matches = list(p.glob("**/valid_labeled_comments.csv"))
        if matches:
            full_df = pd.read_csv(matches[0], encoding="utf-8-sig")
            data_dir = matches[0].parent
            break
        matches_tr = list(p.glob("**/train.csv"))
        if matches_tr:
            data_dir = matches_tr[0].parent
            df_tr = pd.read_csv(data_dir / "train.csv", encoding="utf-8-sig")
            df_va = pd.read_csv(data_dir / "val.csv", encoding="utf-8-sig")
            full_df = pd.concat([df_tr, df_va], ignore_index=True)
            break

if full_df is None:
    raise FileNotFoundError("❌ Tidak dapat menemukan berkas dataset terlabel! Harap upload dataset ke Kaggle.")

print(f"✅ Total Pure Human Gold Standard Dataset: {len(full_df)} baris (dari {data_dir.resolve()})")

# Create Stratification Key for Multi-Aspect Stratified K-Fold
aspect_cols = [f"{a}_sentiment" for a in ASPECTS]
full_df['strat_key'] = (
    full_df[aspect_cols[0]].fillna('none').astype(str) + '_' +
    full_df[aspect_cols[1]].fillna('none').astype(str) + '_' +
    full_df[aspect_cols[2]].fillna('none').astype(str) + '_' +
    full_df[aspect_cols[3]].fillna('none').astype(str)
)

counts = full_df['strat_key'].value_counts()
rare_keys = set(counts[counts < 5].index)
full_df['strat_label'] = full_df['strat_key'].apply(lambda x: 'rare_comb' if x in rare_keys else x)

display(full_df.head(3))


## 3. Focal Loss, Class Weights & ABSADataset

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.5):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        probs = F.softmax(inputs, dim=-1)
        pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        pt = torch.clamp(pt, min=1e-8, max=1.0)
        
        focal_weight = (1.0 - pt) ** self.gamma
        if self.alpha is not None:
            alpha_weight = self.alpha.to(inputs.device)[targets]
            focal_weight = focal_weight * alpha_weight

        loss = -focal_weight * torch.log(pt)
        return loss.mean()

def compute_aspect_class_weights(df, aspects=ASPECTS, smooth_factor=0.5):
    weights_dict = {}
    classes = np.array([0, 1, 2, 3])
    for aspect in aspects:
        col = f"{aspect}_sentiment"
        raw_vals = df[col].fillna("none").astype(str).str.lower().map(LABEL2ID).fillna(0).astype(int).values
        raw_weights = compute_class_weight(class_weight="balanced", classes=classes, y=raw_vals)
        smoothed = np.power(raw_weights, smooth_factor)
        smoothed = smoothed / np.mean(smoothed)
        weights_dict[aspect] = torch.tensor(smoothed, dtype=torch.float32).to(device)
    return weights_dict

class ABSADataset(Dataset):
    def __init__(self, df, tokenizer, max_len=128, aspects=ASPECTS):
        if "text_cleaned" in df.columns:
            text_col = "text_cleaned"
        elif "text_original" in df.columns:
            text_col = "text_original"
        elif "comment_text" in df.columns:
            text_col = "comment_text"
        else:
            raise KeyError(f"❌ Tidak dapat menemukan kolom teks di DataFrame. Kolom tersedia: {list(df.columns)}")

        self.texts = df[text_col].fillna("").astype(str).tolist()
        self.labels = {}
        for aspect in aspects:
            col = f"{aspect}_sentiment"
            raw = df[col].fillna("none").astype(str).str.lower().map(LABEL2ID).fillna(0).astype(int).values
            self.labels[aspect] = torch.tensor(raw, dtype=torch.long)
        
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.aspects = aspects

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        encoding = self.tokenizer(
            text,
            truncation=True,
            max_length=self.max_len,
            padding="max_length",
            return_tensors="pt"
        )
        
        item = {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": {aspect: self.labels[aspect][idx] for aspect in self.aspects}
        }
        return item


## 4. Decision Thresholding & Evaluator Functions

In [ ]:
def predict_with_threshold(logits, none_threshold=ASPECT_THRESHOLDS, aspect=None):
    if isinstance(none_threshold, dict) and aspect in none_threshold:
        thresh = none_threshold[aspect]
    elif isinstance(none_threshold, (int, float)):
        thresh = float(none_threshold)
    else:
        thresh = 0.50

    if isinstance(logits, torch.Tensor):
        probs = F.softmax(logits, dim=-1).detach().cpu().numpy()
    else:
        exp_logits = np.exp(logits - np.max(logits, axis=-1, keepdims=True))
        probs = exp_logits / np.sum(exp_logits, axis=-1, keepdims=True)

    preds = []
    for i in range(probs.shape[0]):
        p_none = probs[i, 0]
        if p_none > thresh:
            preds.append(0)
        else:
            active_probs = probs[i, 1:]
            best_active_idx = int(np.argmax(active_probs)) + 1
            preds.append(best_active_idx)

    return np.array(preds)

def evaluate_predictions(y_true_dict, y_pred_dict, aspects=ASPECTS):
    aspect_metrics = {}
    cm_dict = {}
    macro_f1s_all, macro_f1s_active, accuracies = [], [], []

    for aspect in aspects:
        y_true = np.array(y_true_dict[aspect])
        y_pred = np.array(y_pred_dict[aspect])

        acc = accuracy_score(y_true, y_pred)
        f1_m_all = f1_score(y_true, y_pred, average="macro", zero_division=0)
        f1_m_active = f1_score(y_true, y_pred, labels=[1, 2, 3], average="macro", zero_division=0)

        prec_m = precision_score(y_true, y_pred, average="macro", zero_division=0)
        rec_m = recall_score(y_true, y_pred, average="macro", zero_division=0)
        f1_w = f1_score(y_true, y_pred, average="weighted", zero_division=0)

        cm = confusion_matrix(y_true, y_pred, labels=[0, 1, 2, 3])

        aspect_metrics[aspect] = {
            "accuracy": acc,
            "f1_macro_all": f1_m_all,
            "f1_macro_active": f1_m_active,
            "precision_macro": prec_m,
            "recall_macro": rec_m,
            "f1_weighted": f1_w,
        }
        cm_dict[aspect] = cm
        accuracies.append(acc)
        macro_f1s_all.append(f1_m_all)
        macro_f1s_active.append(f1_m_active)

    n_samples = len(y_true_dict[aspects[0]])
    exact_matches = sum(
        all(y_true_dict[asp][i] == y_pred_dict[asp][i] for asp in aspects)
        for i in range(n_samples)
    )
    exact_match_ratio = exact_matches / n_samples if n_samples > 0 else 0.0

    return {
        "per_aspect": aspect_metrics,
        "overall": {
            "mean_accuracy": float(np.mean(accuracies)),
            "mean_macro_f1_all": float(np.mean(macro_f1s_all)),
            "mean_macro_f1_active": float(np.mean(macro_f1s_active)),
            "exact_match_ratio": float(exact_match_ratio),
        },
        "confusion_matrices": cm_dict,
    }

def plot_confusion_matrices(cm_dict, model_name, save_path, aspects=ASPECTS):
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    fig.suptitle(f"Confusion Matrices 4x4 — {model_name}", fontsize=16, fontweight="bold", y=0.98)

    for idx, aspect in enumerate(aspects):
        ax = axes[idx // 2, idx % 2]
        cm = cm_dict[aspect]
        sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
        ax.set_title(f"Aspek: {aspect.capitalize()}", fontsize=14, fontweight="semibold")
        ax.set_xlabel("Predicted Label", fontsize=11)
        ax.set_ylabel("True Label", fontsize=11)

    plt.tight_layout(rect=[0, 0, 1, 0.95])
    plt.savefig(save_path, dpi=300, bbox_inches="tight")
    plt.show()
    plt.close()
    return save_path


## 5. PyTorch Multi-Head Classifier Model Classes

In [ ]:
class IndoRoBERTaMultiHeadClassifier(nn.Module):
    def __init__(self, model_name="indolem/indobert-base-uncased", num_classes=4, dropout_prob=0.1, aspects=ASPECTS, focal_gamma=1.5):
        super().__init__()
        self.aspects = aspects
        self.focal_gamma = focal_gamma
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.heads = nn.ModuleDict({
            aspect: nn.Sequential(
                nn.Linear(hidden_size * 2, 256),
                nn.GELU(),
                nn.Dropout(dropout_prob),
                nn.Linear(256, num_classes)
            ) for aspect in aspects
        })

    def forward(self, input_ids, attention_mask, labels=None, class_weights=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state

        cls_token = last_hidden[:, 0, :]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_embeddings = torch.sum(last_hidden * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        pooled_features = torch.cat([cls_token, mean_pooled], dim=-1)

        logits = {}
        total_loss = 0.0

        for aspect in self.aspects:
            aspect_logits = self.heads[aspect](pooled_features)
            logits[aspect] = aspect_logits

            if labels is not None:
                target = labels[aspect].to(aspect_logits.device)
                alpha = class_weights[aspect] if class_weights and aspect in class_weights else None
                criterion = FocalLoss(alpha=alpha, gamma=self.focal_gamma)
                total_loss += criterion(aspect_logits, target)

        output_dict = {"logits": logits}
        if labels is not None:
            output_dict["loss"] = total_loss
        return output_dict

class XLMRoBERTaMultiHeadClassifier(nn.Module):
    def __init__(self, model_name="xlm-roberta-large", num_classes=4, dropout_prob=0.1, aspects=ASPECTS, focal_gamma=1.5):
        super().__init__()
        self.aspects = aspects
        self.focal_gamma = focal_gamma
        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size  # 1024

        # Expanded 2-Layer GELU MLP Head with LayerNorm (2048 -> 512 -> 4)
        self.heads = nn.ModuleDict({
            aspect: nn.Sequential(
                nn.Linear(hidden_size * 2, 512),
                nn.LayerNorm(512),
                nn.GELU(),
                nn.Dropout(dropout_prob),
                nn.Linear(512, num_classes)
            ) for aspect in aspects
        })

    def forward(self, input_ids, attention_mask, labels=None, class_weights=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden = outputs.last_hidden_state

        cls_token = last_hidden[:, 0, :]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden.size()).float()
        sum_embeddings = torch.sum(last_hidden * input_mask_expanded, dim=1)
        sum_mask = torch.clamp(input_mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = sum_embeddings / sum_mask

        pooled_features = torch.cat([cls_token, mean_pooled], dim=-1)

        logits = {}
        total_loss = 0.0

        for aspect in self.aspects:
            aspect_logits = self.heads[aspect](pooled_features)
            logits[aspect] = aspect_logits

            if labels is not None:
                target = labels[aspect].to(aspect_logits.device)
                alpha = class_weights[aspect] if class_weights and aspect in class_weights else None
                criterion = FocalLoss(alpha=alpha, gamma=self.focal_gamma)
                total_loss += criterion(aspect_logits, target)

        output_dict = {"logits": logits}
        if labels is not None:
            output_dict["loss"] = total_loss
        return output_dict


## 6. 5-Fold Stratified Cross-Validation (Single Best Model Disk Safeguard)

In [ ]:
def train_5fold_cv(model_class, model_name, tokenizer, full_df, epochs=20, lr=2e-5, batch_size=16, grad_accum=1, save_prefix="model"):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_metrics = []
    os.makedirs(f"models/{save_prefix}", exist_ok=True)

    best_overall_val_f1 = 0.0
    best_model_path = f"models/{save_prefix}/pytorch_model.bin"

    print(f"
🚀 STARTING 5-FOLD CROSS-VALIDATION: {model_name} (20 Epochs per fold)")
    print(f"💾 DISK SAFEGUARD: Hanya menyimpan 1 Single Best Model Checkpoint ({save_prefix}/pytorch_model.bin)")

    for fold, (train_idx, val_idx) in enumerate(skf.split(full_df, full_df['strat_label']), 1):
        print(f"
========================================================")
        print(f"🔄 FOLD {fold}/5 — {model_name}")
        print(f"========================================================")

        train_fold_df = full_df.iloc[train_idx].reset_index(drop=True)
        val_fold_df = full_df.iloc[val_idx].reset_index(drop=True)

        class_weights = compute_aspect_class_weights(train_fold_df)

        train_dataset = ABSADataset(train_fold_df, tokenizer, max_len=128)
        val_dataset = ABSADataset(val_fold_df, tokenizer, max_len=128)

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model = model_class(model_name).to(device)

        total_steps = (len(train_loader) // grad_accum) * epochs
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
        scheduler = get_cosine_schedule_with_warmup(
            optimizer,
            num_warmup_steps=int(0.1 * total_steps),
            num_training_steps=total_steps
        )

        best_fold_f1_active = 0.0
        best_fold_eval = None

        for epoch in range(1, epochs + 1):
            model.train()
            total_train_loss = 0.0
            optimizer.zero_grad()

            for step, batch in enumerate(train_loader):
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = {k: v.to(device) for k, v in batch["labels"].items()}

                output = model(input_ids, attention_mask, labels=labels, class_weights=class_weights)
                loss = output["loss"] / grad_accum
                loss.backward()

                if (step + 1) % grad_accum == 0 or (step + 1) == len(train_loader):
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

                total_train_loss += loss.item() * grad_accum

            avg_train_loss = total_train_loss / len(train_loader)

            # Evaluate Val F1 for EVERY epoch
            model.eval()
            y_true_epoch = {asp: [] for asp in ASPECTS}
            y_pred_epoch = {asp: [] for asp in ASPECTS}

            with torch.no_grad():
                for batch in val_loader:
                    input_ids = batch["input_ids"].to(device)
                    attention_mask = batch["attention_mask"].to(device)
                    labels = batch["labels"]

                    output = model(input_ids, attention_mask)
                    logits = output["logits"]

                    for asp in ASPECTS:
                        preds = predict_with_threshold(logits[asp], aspect=asp)
                        y_pred_epoch[asp].extend(preds)
                        y_true_epoch[asp].extend(labels[asp].numpy())

            epoch_eval = evaluate_predictions(y_true_epoch, y_pred_epoch)
            val_f1_act = epoch_eval["overall"]["mean_macro_f1_active"]
            val_f1_all = epoch_eval["overall"]["mean_macro_f1_all"]
            val_acc = epoch_eval["overall"]["mean_accuracy"]

            # REAL-TIME LOG DISPLAY FOR EVERY EPOCH
            print(f"  Epoch {epoch:2d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val F1 (Active): {val_f1_act:.4f} | Val F1 (All): {val_f1_all:.4f} | Val Acc: {val_acc:.4f}")

            if val_f1_act > best_fold_f1_active:
                best_fold_f1_active = val_f1_act
                best_fold_eval = epoch_eval

            # Check if this is the overall single best model across ALL folds
            if val_f1_act > best_overall_val_f1:
                best_overall_val_f1 = val_f1_act
                torch.save(model.state_dict(), best_model_path)
                print(f"  🌟 NEW BEST OVERALL MODEL SAVED! (Val F1 Active: {val_f1_act:.4f})")

        print(f"⭐ Fold {fold} Best Val F1 (Active): {best_fold_f1_active:.4f}")
        fold_metrics.append(best_fold_eval)

        # Explicitly release VRAM and memory per fold
        del model, optimizer, scheduler
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    tokenizer.save_pretrained(f"models/{save_prefix}")
    print(f"
🏆 SUCCESS: 5-Fold CV Completed for {model_name}!")
    print(f"💾 Single Best Model Saved at: {best_model_path} (Best F1 Active: {best_overall_val_f1:.4f})")
    return fold_metrics


## 7. Model 1: IndoRoBERTa-base (110M) 5-Fold Stratified CV

In [ ]:
indobert_model_name = "indolem/indobert-base-uncased"
indobert_tokenizer = AutoTokenizer.from_pretrained(indobert_model_name)

roberta_fold_metrics = train_5fold_cv(
    model_class=IndoRoBERTaMultiHeadClassifier,
    model_name=indobert_model_name,
    tokenizer=indobert_tokenizer,
    full_df=full_df,
    epochs=20,
    lr=2e-5,
    batch_size=16,
    grad_accum=1,
    save_prefix="indoroberta_absa"
)

roberta_f1_active_folds = [m["overall"]["mean_macro_f1_active"] for m in roberta_fold_metrics]
roberta_f1_all_folds = [m["overall"]["mean_macro_f1_all"] for m in roberta_fold_metrics]
roberta_acc_folds = [m["overall"]["mean_accuracy"] for m in roberta_fold_metrics]

print("
📊 HASIL EVALUASI 5-FOLD CV — INDOROBERTA-BASE (110M):")
print(f"  • Mean Macro F1 (Active Sentiments): {np.mean(roberta_f1_active_folds):.4f} ± {np.std(roberta_f1_active_folds):.4f}")
print(f"  • Mean Macro F1 (All Classes)      : {np.mean(roberta_f1_all_folds):.4f} ± {np.std(roberta_f1_all_folds):.4f}")
print(f"  • Mean Accuracy                  : {np.mean(roberta_acc_folds):.4f} ± {np.std(roberta_acc_folds):.4f}")


## 8. Model 2: XLM-RoBERTa-large (550M) 5-Fold Stratified CV

In [ ]:
xlm_model_name = "xlm-roberta-large"
xlm_tokenizer = AutoTokenizer.from_pretrained(xlm_model_name)

xlm_fold_metrics = train_5fold_cv(
    model_class=XLMRoBERTaMultiHeadClassifier,
    model_name=xlm_model_name,
    tokenizer=xlm_tokenizer,
    full_df=full_df,
    epochs=20,
    lr=1.5e-5,
    batch_size=8,
    grad_accum=2,
    save_prefix="xlmroberta_absa"
)

xlm_f1_active_folds = [m["overall"]["mean_macro_f1_active"] for m in xlm_fold_metrics]
xlm_f1_all_folds = [m["overall"]["mean_macro_f1_all"] for m in xlm_fold_metrics]
xlm_acc_folds = [m["overall"]["mean_accuracy"] for m in xlm_fold_metrics]

print("
📊 HASIL EVALUASI 5-FOLD CV — XLM-ROBERTA-LARGE (550M):")
print(f"  • Mean Macro F1 (Active Sentiments): {np.mean(xlm_f1_active_folds):.4f} ± {np.std(xlm_f1_active_folds):.4f}")
print(f"  • Mean Macro F1 (All Classes)      : {np.mean(xlm_f1_all_folds):.4f} ± {np.std(xlm_f1_all_folds):.4f}")
print(f"  • Mean Accuracy                  : {np.mean(xlm_acc_folds):.4f} ± {np.std(xlm_acc_folds):.4f}")


## 9. Comparative Analysis & Summary Metrics (5-Fold CV Benchmark)

In [ ]:
overall_summary = pd.DataFrame([
    {
        "Model": "IndoRoBERTa-base Classifier (110M)",
        "Mean Macro F1 (Active)": f"{np.mean(roberta_f1_active_folds):.4f} ± {np.std(roberta_f1_active_folds):.4f}",
        "Mean Macro F1 (All)": f"{np.mean(roberta_f1_all_folds):.4f} ± {np.std(roberta_f1_all_folds):.4f}",
        "Mean Accuracy": f"{np.mean(roberta_acc_folds):.4f} ± {np.std(roberta_acc_folds):.4f}",
    },
    {
        "Model": "XLM-RoBERTa-large Classifier (550M)",
        "Mean Macro F1 (Active)": f"{np.mean(xlm_f1_active_folds):.4f} ± {np.std(xlm_f1_active_folds):.4f}",
        "Mean Macro F1 (All)": f"{np.mean(xlm_f1_all_folds):.4f} ± {np.std(xlm_f1_all_folds):.4f}",
        "Mean Accuracy": f"{np.mean(xlm_acc_folds):.4f} ± {np.std(xlm_acc_folds):.4f}",
    }
])

print("🌟 RINGKASAN PERBANDINGAN GLOBAL (5-FOLD STRATIFIED CV):")
display(overall_summary)

overall_summary.to_csv("model_comparison_metrics_5fold.csv", index=False)

summary_json = {
    "indoroberta_5fold": {
        "mean_f1_active": float(np.mean(roberta_f1_active_folds)),
        "std_f1_active": float(np.std(roberta_f1_active_folds)),
        "folds": roberta_fold_metrics
    },
    "xlmroberta_5fold": {
        "mean_f1_active": float(np.mean(xlm_f1_active_folds)),
        "std_f1_active": float(np.std(xlm_f1_active_folds)),
        "folds": xlm_fold_metrics
    }
}
with open("model_comparison_metrics_5fold.json", "w", encoding="utf-8") as f:
    json.dump(summary_json, f, indent=2, default=str)

print("✅ Metrik 5-Fold Cross Validation berhasil disimpan ke file CSV & JSON.")


## 10. Zip & Package Deployment Artifacts for Download

In [ ]:
def zip_dir(source_dir, zip_filename):
    with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as ziph:
        for root, dirs, files in os.walk(source_dir):
            for file in files:
                filepath = os.path.join(root, file)
                arcname = os.path.relpath(filepath, source_dir)
                ziph.write(filepath, arcname)
    print(f"📦 Zipped '{source_dir}' -> '{zip_filename}' ({os.path.getsize(zip_filename) / 1e6:.2f} MB)")

zip_dir("models/indoroberta_absa", "indoroberta_absa_model.zip")
zip_dir("models/xlmroberta_absa", "xlmroberta_absa_model.zip")

print("
🎉 SELESAI! Berkas berikut siap diunduh dari Kaggle Output:")
print("  1. indoroberta_absa_model.zip (~440 MB - Single Best Model)")
print("  2. xlmroberta_absa_model.zip (~2.1 GB - Single Best Model)")
print("  3. model_comparison_metrics_5fold.csv & JSON")
